In [3]:
import pandas as pd
import numpy as np
import re

# =====================================================
# LOAD
# =====================================================
df = pd.read_csv("crm_output.csv", dtype=str)
out = df.copy()

# =====================================================
# HELPERS
# =====================================================
def col(name):
    return df[name] if name in df.columns else pd.Series(np.nan, index=df.index)

def clean_phone(x):
    if pd.isna(x): return np.nan
    d = "".join(c for c in str(x) if c.isdigit())
    if d.startswith("380") and len(d) == 12: return "+" + d
    if d.startswith("0") and len(d) == 10: return "+38" + d
    return "+" + d if d else np.nan

clean_email = lambda x: np.nan if pd.isna(x) else str(x).strip().lower()

def clean_money(x):
    if pd.isna(x): return np.nan
    x = re.sub(r"[₴грн\s]", "", str(x)).replace(",", ".")
    x = re.sub(r"[^0-9.]", "", x)
    try: return float(x)
    except: return np.nan

def extract_phone(x):
    if pd.isna(x): return np.nan
    m = re.findall(r'[\+\d\-\(\)\s]{10,20}', str(x))
    return clean_phone(m[0]) if m else np.nan

# =====================================================
# NOTES
# =====================================================
note_cols = [c for c in ["Note","Note 2","Note 3","Note 4","Note 5"] if c in df.columns]
out["All_Notes"] = df[note_cols].fillna("").agg(" ".join, axis=1)

# =====================================================
# TELEGRAM
# =====================================================
def extract_telegram(x):
    if pd.isna(x): return np.nan
    for p in [r'@([A-Za-z0-9_]{5,32})', r't\.me/([A-Za-z0-9_]{5,32})']:
        m = re.search(p, str(x), re.I)
        if m: return "@" + m.group(1)
    return np.nan

out["Telegram"] = (
    col("Telegram")
    .fillna(col("Telegram username"))
    .fillna(col("TG"))
    .fillna(out["All_Notes"].apply(extract_telegram))
)

# =====================================================
# CONTACTS
# =====================================================
phone_notes = out["All_Notes"].apply(extract_phone)

out["Clean_Phone"] = (
    col("Mobile phone")
    .fillna(col("Work phone"))
    .fillna(col("Work DD phone"))
    .fillna(col("Home phone"))
    .fillna(col("Other phone"))
    .fillna(phone_notes)
    .apply(clean_phone)
)

out["Clean_Email"] = (
    col("Work email")
    .fillna(col("Personal email"))
    .fillna(col("Other email"))
    .apply(clean_email)
)

# =====================================================
# COURSE
# =====================================================
out["Clean_Course"] = (
    col("Lead title")
    .fillna(col("Курс"))
    .fillna(col("COURSE"))
    .fillna(col("Название пакета"))
    .fillna(out["All_Notes"])
    .fillna("(no-course)")
)

# =====================================================
# FINANCE
# =====================================================
out["Clean_Sale"] = col("Lead sale ₴").apply(clean_money).fillna(0)

out["Clean_Payment"] = (
    col("Оплата")
    .fillna(out["All_Notes"].str.extract(r'(\d+[.,]?\d*)')[0])
    .apply(clean_money)
    .fillna(0)
)

# =====================================================
# UTM (compact merge)
# =====================================================
def merge(cols):
    return df.apply(
        lambda r: " | ".join(
            dict.fromkeys(
                str(r[c]).strip()
                for c in cols
                if c in df.columns and pd.notna(r[c]) and str(r[c]).strip() and str(r[c]).lower() != "nan"
            )
        ) or "(not set)",
        axis=1
    ).str.lower()

out["UTM_Source"]   = merge(["utm_source","utm_source.1","TRAF_SRC","openstat_source"])
out["UTM_Medium"]   = merge(["utm_medium","TRAF_TYPE","utm_medium.1"])
out["UTM_Campaign"] = merge(["utm_campaign","utm_campaign","ADV_CAMP","openstat_campaign"])
out["UTM_Term"]     = merge(["utm_term","utm_term.1"])
out["UTM_Content"]  = merge(["utm_content","utm_content.1"])
out["Referrer"]     = merge(["Referrer","REFERER","referrer","utm_referrer"])

# =====================================================
# IDS + QUIZ
# =====================================================
out["Clean_Gclid"] = col("gclid").fillna(col("gclientid")).fillna(col("GOOGLE_ID"))

quiz_cols = [c for c in ["Вопрос 1","Вопрос 2","Вопрос 3","Вопрос 4","Вопрос 5"] if c in df.columns]

out["Quiz_Summary"] = df[quiz_cols].apply(
    lambda r: " | ".join(x.strip() for x in r if pd.notna(x) and str(x).strip()),
    axis=1
).replace("", np.nan)

# =====================================================
# DATES
# =====================================================
closed = col("Closed at").replace("not closed", np.nan)

out["Closed_At"] = pd.to_datetime(
    closed,
    format="%d.%m.%Y %H:%M:%S",
    errors="coerce"
)

out["Created_On"] = pd.to_datetime(col("Created on"), errors="coerce")

# =====================================================
# META
# =====================================================
out["Lead_ID"] = col("ID")
out["Lead_Stage"] = col("Lead stage")
out["Pipeline"] = col("Pipeline")

# =====================================================
# DUPLICATES
# =====================================================
out["Lead_Key"] = (
    out["Clean_Phone"]
    .replace("", np.nan)
    .fillna(out["Clean_Email"])
    .fillna(out["Lead_ID"])
)

out["Is_Duplicate"] = out.duplicated(["Lead_Key"], keep="first")

# =====================================================
# CLEANUP
# =====================================================
out = out.drop(columns=[c for c in df.columns if c not in out.columns and "utm" in c], errors="ignore")

def remove_garbage(df, thr=0.95):
    df = df.replace([r"^\s*$","nan","null","none"], np.nan, regex=True)
    return df.loc[:, ~(df.isna().mean() > thr)]

out = remove_garbage(out)

# =====================================================
# EXPORT
# =====================================================
out.to_csv("powerbi_leads_clean.csv", index=False, encoding="utf-8-sig")

print("DONE:", out.shape)

C:\Users\User\AppData\Local\Temp\ipykernel_22504\4044833215.py:179: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace([r"^\s*$","nan","null","none"], np.nan, regex=True)


DONE: (927, 72)
